# ___Correlated evolution between a continuous trait and a discrete categorical trait___
--------------------------------------

In [ ]:
# look up https://thej022214.github.io/OUwie/articles/hOUwieStarterGuide.html

In [1]:
R.version

               _                                
platform       x86_64-w64-mingw32               
arch           x86_64                           
os             mingw32                          
crt            ucrt                             
system         x86_64, mingw32                  
status                                          
major          4                                
minor          5.2                              
year           2025                             
month          10                               
day            31                               
svn rev        88974                            
language       R                                
version.string R version 4.5.2 (2025-10-31 ucrt)
nickname       [Not] Part in a Rumble           

In [2]:
library("ape")
library("phytools")
library("nlme")
library("corHMM")
library("geiger")
library("mkcor")
library("OUwie")

Loading required package: maps

Loading required package: nloptr

Loading required package: GenSA

Loading required package: corpcor

Loading required package: RColorBrewer



In [4]:
packageVersion("OUwie") # make sure it is 2.16 

[1] '2.16'

## ___Model fitting to dummy data___
-----------------

In [16]:
# example model fitting

data(tworegime)
dat <- data.frame(sp = tree$tip.label, X = sample(c(0, 1, 2), length(tree$tip.label), replace = TRUE), Y = sample(c(0, 1), length(tree$tip.label), replace = TRUE), FS = rnorm(length(tree$tip.label), 10, 3))
head(dat)

,sp,X,Y,FS
,<chr>,<dbl>,<dbl>,<dbl>
1,t1,0,1,9.365061
2,t2,0,0,10.502899
3,t3,2,0,5.821208
4,t4,0,0,9.402193
5,t5,1,1,11.637713
6,t6,0,0,8.751180


In [17]:
p <- c(0.01670113, 0.39489947, 0.18619839, 1.67259459, 0.16817414)  # my fixed set of parameters
pp_oum <- OUwie::hOUwie(tree, trait, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25, p = p)  # you likely won't use this p argument

Negative values detected... adding 50 to the trait mean for optimization purposes
Your phylogeny had node labels, these have been removed.
Calculating likelihood from a set of fixed parameters.
[1]  0.01670113  0.39489947  0.18619839 51.67259459 50.16817414


In [17]:
pp_oum


Fit
    lnLTot   lnLDisc   lnLCont     AIC     AICc      BIC nTaxa nPars
 -25.15545 -5.260483 -18.62633 60.3109 61.34539 71.10532    64     5

Legend
  1   2 
"1" "2" 

Regime Rate matrix
           (1)        (2)
(1)         NA 0.01670113
(2) 0.01670113         NA

OU Estimates
             (1)       (2)
alpha  0.3948995 0.3948995
sigma2 0.1861984 0.1861984
theta  1.6725946 0.1681741


Half-life (another way of reporting alpha)
    (1)     (2) 
1.75525 1.75525 

In [6]:
# fitting without p
model <- OUwie::hOUwie(tree, trait, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25) 

Negative values detected... adding 50 to the trait mean for optimization purposes
Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


## ___Fitting the `hOUwie` model to our data___
-------------------------

In [16]:
# fit the model to my data
fred_tree <- ape::multi2di(ape::read.tree("../data/chapter2/uphylomaker/fredv3subset_collab_trait_n_states.tre"))
data <- read.csv("../data/chapter2/FREDv3subset/FRED_subset_collab_states_n_species_avg_traits.csv", row.names = "binominal")

In [17]:
data <- data.frame(binominal = as.factor(gsub(rownames(data), pattern = ' ', replacement = '_')), rd = data$F00679, srl = data$F00727, myco = as.factor(data$F00645))
row_indices <- match(fred_tree$tip.label, data$binominal)
all(data$binominal[row_indices] == fred_tree$tip.label)
data <- data[row_indices, ]
all(data$binominal == fred_tree$tip.label) # cool

[1] TRUE

[1] TRUE

In [18]:
length(unique(data$binominal)) == length(data$binominal) # good, no duplicates

[1] TRUE

In [20]:
levels(data$myco)

[1] "AM"   "AMEM" "AMNM" "EM"   "ErM"  "NM"

In [23]:
head(data)

,binominal,rd,srl,myco
,<fct>,<dbl>,<dbl>,<fct>
39,Anaphalis_aureopunctata,0.136950,479.1550,AMNM
40,Anaphalis_hancockii,0.178700,319.8579,AM
311,Solidago_decurrens,0.248225,202.8837,AM
129,Doellingeria_scabra,0.211300,219.2008,AM
54,Aster_tataricus,0.197000,297.1100,AM
49,Artemisia_igniaria,0.167200,204.1529,AM


In [24]:
# OUwie::hOUwie expects the columns in the following order => species name, categorical trait, continuous trait

data_rd <- data[, c("binominal", "myco", "rd")]
data_srl <- data[, c("binominal", "myco", "srl")]

In [50]:
# BM1 and OU1 models expect no link between the discrete and continuous characters. I.e., all of the parameters are identical for different states of the categorical vector.
# BMV on the other hand suggests that the rates of evolution differs between different states of the categorical vector, as measure by sigma2.
# FUCK R AND ITS GROTESQUE SYNTAX

runtime_OUM <- Sys.time()
model_rd_OUM <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25) # n_starts = 14, ncores = 14) => these args are NOT supported on Windows???
runtime_OUM <- Sys.time() - runtime_OUM

runtime_BM1 <- Sys.time()
model_rd_BM1 <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, discrete_model = "ER", continuous_model = "BM1", nSim = 25)
runtime_BM1 <- Sys.time() - runtime_BM1

runtime_OU1 <- Sys.time()
model_rd_OU1 <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, discrete_model = "ER", continuous_model = "OU1", nSim = 25)
runtime_OU1 <- Sys.time() - runtime_OU1

runtime_BMV <- Sys.time()
model_rd_BMV <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, discrete_model = "ER", continuous_model = "BMV", nSim = 25)
runtime_BMV <- Sys.time() - runtime_BMV

Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


In [53]:
# damn
runtime_BM1
runtime_BMV
runtime_OU1
runtime_OUM

Time difference of 2.611249 mins

Time difference of 20.52963 mins

Time difference of 4.900693 mins

Time difference of 24.8017 mins

In [54]:
# summary of model performances
models <- list(bm1 = model_rd_BM1, bmv = model_rd_BMV, ou1 = model_rd_OU1, oum = model_rd_OUM)
OUwie::getModelTable(models)

,np,lnLik,DiscLik,ContLik,BIC,dBIC,BICwt
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
bm1,3,-216.9088,-171.5257,-42.30647,451.3395,158.30858,4.204467e-35
bmv,8,-203.9080,-171.4943,-28.85857,454.5411,161.51017,8.481950e-36
ou1,4,-134.8342,-171.5320,39.79815,293.0309,0.00000,9.999602e-01
oum,9,-130.3638,-171.4534,44.27947,313.2934,20.26246,3.981482e-05


In [ ]:
# allowing rate and optima variation

runtime_OUM_null <- Sys.time()
model_rd_OUM_null <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, discrete_model = "ER", continuous_model = "OUM", nSim = 25, null.model = TRUE)
runtime_OUM_null <- Sys.time() - runtime_OUM_null

runtime_BM1_null <- Sys.time()
model_rd_BM1_null <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, discrete_model = "ER", continuous_model = "BM1", nSim = 25, null.model = TRUE)
runtime_BM1_null <- Sys.time() - runtime_BM1_null

runtime_OU1_null <- Sys.time()
model_rd_OU1_null <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, discrete_model = "ER", continuous_model = "OU1", nSim = 25, null.model = TRUE)
runtime_OU1_null <- Sys.time() - runtime_OU1_null

runtime_BMV_null <- Sys.time()
model_rd_BMV_null <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, discrete_model = "ER", continuous_model = "BMV", nSim = 25, null.model = TRUE)
runtime_BMV_null <- Sys.time() - runtime_BMV_null

Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


In [ ]:
runtime_BM1_null
runtime_BMV_null
runtime_OU1_null
runtime_OUM_null

In [ ]:
# serialize all the models, we DO NOT WANT TO SEPND ANOTHER THREE HOURS, repeating this shite

saveRDS(object = model_rd_OUM, file = "../data/chapter2/hOUwie/houwie_rd_OUM.rds")
saveRDS(object = model_rd_BM1, file = "../data/chapter2/hOUwie/houwie_rd_BM1.rds")
saveRDS(object = model_rd_OU1, file = "../data/chapter2/hOUwie/houwie_rd_OU1.rds")
saveRDS(object = model_rd_BMV, file = "../data/chapter2/hOUwie/houwie_rd_BMV.rds")

saveRDS(object = model_rd_OUM_null, file = "../data/chapter2/hOUwie/houwie_rd_OUM_null.rds")
saveRDS(object = model_rd_BM1_null, file = "../data/chapter2/hOUwie/houwie_rd_BM1_null.rds")
saveRDS(object = model_rd_OU1_null, file = "../data/chapter2/hOUwie/houwie_rd_OU1_null.rds")
saveRDS(object = model_rd_BMV_null, file = "../data/chapter2/hOUwie/houwie_rd_BMV_null.rds")